In [11]:
import sys
import os

# sobe um nível (da pasta notebooks → raiz do projeto)
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)


In [12]:
from pipeline.context import ClaimContext
from pipeline.pipeline import Pipeline

from modules.llm.ollama_interface import OllamaLLM
from modules.question_generation.question_generator import QuestionGenerator
from modules.search.web_search import WebSearch
from modules.parsing.document_parser import DocumentParser
from modules.segmentation.passage_extractor import PassageExtractor
from modules.retrieval.bm25_retriever import BM25Retriever

from modules.verdict.rule_verdict import RuleVerdict
from modules.verdict.majority_verdict import MajorityVerdict
from modules.verdict.weighted_verdict import WeightedVerdict
from modules.verdict.llm_verdict import LLMVerdict

from modules.stance.llm_stance_detector import LLMStanceDetector
from modules.reranking.cross_encoder_reranker import CrossEncoderReranker
from modules.qa.qa_generator import QAGenerator
from modules.search.web_search import WebSearch
from modules.justification.justification_generator import JustificationGenerator

from dotenv import load_dotenv

In [13]:
load_dotenv()

searcher = WebSearch(api_key=os.getenv("BRAVE_API_KEY"))
llm = OllamaLLM()

In [14]:
pipeline = Pipeline(
    question_generator=QuestionGenerator(llm),
    searcher=searcher,
    parser=DocumentParser(),
    segmenter=PassageExtractor(),
    retriever=BM25Retriever(),
    qa_generator=QAGenerator(llm),
    stance_detector=LLMStanceDetector(llm),
    verdict_predictor=MajorityVerdict(),
    justification_generator=JustificationGenerator(llm),
    reranker=CrossEncoderReranker()
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 655.15it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
context = ClaimContext(
    claim_id=1,
    claim_text="Hunter Biden had no experience in the energy sector before Burisma."
)

In [16]:
result = pipeline.run(context)

CACHE HIT: Hunter Biden had no experience in the energy sector before Burisma.
API SEARCH: 1. Was Hunter Biden working on a consulting contract with a Chinese energy company called CEFC in 2015?
API SEARCH: 2. Did Hunter Biden join the board of directors at Burisma Holdings in April 2014?
API SEARCH: 3. Before joining Burisma, did Hunter Biden have any professional experience or connections to the Ukrainian energy sector?
PAGE CACHE HIT: https://www.rferl.org/a/burisma-ukraine-gas-pardon-bidens/33226807.html
PAGE CACHE HIT: https://en.wikipedia.org/wiki/Hunter_Biden


In [18]:
print("\nCLAIM:")
print(result.claim)

print("\nTOP EVIDENCE:")

for evidence in result.evidence:
    print("-", evidence)

print("\nSTANCE CLASSIFICATION:")

for evidence, stance in result.stances:
    print(f"[{stance}] {evidence}")

print("\nQA PAIRS:\n")

for qa in result.qa_pairs:

    print("Q:", qa["question"])
    print("A:", qa["answer"])
    print()

print("\nFINAL VERDICT:")
print(result.verdict.upper())

print("\nJUSTIFICATION:\n")
print(result.justification)


CLAIM:
Hunter Biden had no experience in the energy sector before Burisma.

TOP EVIDENCE:
- earlier, his father had visited Ukraine and become the “public face” of the Obama administration’s policy in the region. Hunter was paid as much as $50,000 a month, according to Senate records, despite having no experience in Ukraine or the energy industry, and Burisma stated he would provide assistance with “transparency, corporate governance and responsibility, international expansion and other priorities.” An email obtained by The New York Post purportedly sent to Hunter and business partner Devon Archer on May 12 by Vadym Pozharskyi, an adviser to the board of Burisma, asked for advice on how Hunter could “use your influence”
- control of Ukraine's Crimean Peninsula. Burisma's owner, Mykola Zlochevskiy, a former environment and natural resources minister, was under investigation for corruption, accused of having used his position to acquire lucrative gas fields. Hunter Biden had no backgrou